# Imports

In [1]:
import re
import optuna
import pickle

import numpy as np
import pandas as pd

from sklearn import set_config
from category_encoders import CatBoostEncoder

from sklearn.metrics import roc_auc_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import TargetEncoder, StandardScaler, RobustScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold

from feature_engine.selection import DropFeatures

/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Utils

In [2]:
set_config(transform_output="pandas")

In [3]:
def dump_pickle(file_obj, file_path):
    with open(file_path, 'bw') as file:
        pickle.dump(file_obj, file)

In [4]:
column_transformer = ColumnTransformer([
    (
        'target_encoder', 
        TargetEncoder(), 
        ['driver', 'compound', 'race']
    ),
    (
        'catboost_encoder', 
        CatBoostEncoder(), 
        ['driver', 'compound', 'race']
    ),
    (
        'standard_scaler', 
        StandardScaler(), 
        ['lapnumber', 'position', 'raceprogress', 'year', 'position_norm', 'race_progress_sin', 'position_vs_mean']
    ),
    (
        'robust_scaler', 
        RobustScaler(), 
        [
            'position_change', 'cumulative_degradation', 'laptime_delta', 'laptime_s', 'stint', 'driver_mean_lap', 'tyrelife', 'delta_x_tyre_life', 
            'compound_tyre_life', 'stint_progress', 'tyre_life_ratio', 'degradation_per_lap', 'position_change_cum', 'laps_since_pit', 'lap_time_inv',  
            'lap_time_vs_race_mean', 'lap_time_x_tyre', 'position_x_progress', 'degradation_x_progress', 'race_progress_squared', 'driver_avg_position' 
        ]
    ),
], remainder="passthrough")

# Loading Dataset

In [5]:
X_train = pd.read_parquet('../data/X_train.parquet')
y_train = pd.read_parquet('../data/y_train.parquet')

In [6]:
X_train.head()

,driver,compound,race,year,pitstop,lapnumber,stint,tyrelife,position,laptime_s,...,treeraceprogress,treeposition,treelaptime_s,treecumulative_degradation,treeraceprogress_position,treeraceprogress_laptime_s,treeraceprogress_cumulative_degradation,treeposition_laptime_s,treeposition_cumulative_degradation,treelaptime_s_cumulative_degradation
id,,,,,,,,,,,,,,,,,,,,,
0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,...,0.298107,0.207392,0.252153,0.279021,0.298107,0.298107,0.321361,0.252153,0.279021,0.253721
1,D086,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,...,0.298107,0.185508,0.252153,0.586119,0.298107,0.298107,0.610176,0.252153,0.544023,0.613159
2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,...,0.298107,0.207392,0.252153,0.207676,0.298107,0.298107,0.423519,0.252153,0.228194,0.231339
3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,...,0.102471,0.185508,0.178575,0.138352,0.102471,0.102471,0.043550,0.178575,0.138352,0.070991
4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,...,0.298107,0.185508,0.178575,0.138352,0.298107,0.298107,0.206543,0.178575,0.138352,0.153276


# Machine Learning

## Base

In [7]:
pipe_model = make_pipeline(
    column_transformer,
    LogisticRegression(solver="lbfgs", max_iter=2000, class_weight="balanced")
)

In [8]:
cv_results = cross_val_score(
    estimator=pipe_model, 
    X=X_train, 
    y=y_train.PitNextLap, 
    scoring='roc_auc', 
    cv=StratifiedKFold(random_state=42, shuffle=True), 
    n_jobs=11
)

In [9]:
cv_results.mean()

np.float64(0.8932782421408249)

In [10]:
model = pipe_model.fit(X_train, y_train.PitNextLap)

## Feature Selection

In [11]:
perm_result = permutation_importance(
    estimator=model, 
    X=X_train, 
    y=y_train.PitNextLap, 
    n_jobs=11, 
    scoring='roc_auc'
)

importance_df = pd.DataFrame({
    "feature": X_train.columns.tolist(),
    "importance_mean": perm_result.importances_mean,
    "importance_std": perm_result.importances_std
}).sort_values(by="importance_mean", ascending=False)

In [12]:
importance_df.query("importance_mean <= 0").feature.tolist()

['lapnumber_high']

In [13]:
features_to_drop = importance_df.query("importance_mean <= 0").feature.tolist()

## Fine Tuning

In [14]:
def objective(trial, X, y):
    
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    aucs = []

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y)):
        X_train, X_valid = X.iloc[train_idx, :], X.iloc[valid_idx, :]
        y_train, y_valid = y.iloc[train_idx, 0], y.iloc[valid_idx, 0]

        model = make_pipeline(
            column_transformer,
            LogisticRegression(
                C=trial.suggest_float("C", 1e-3, 10, log=True),
                class_weight="balanced",
                max_iter=2000,
                random_state=42
            )
        ).fit(X_train, y_train)
        
        proba = model.predict_proba(X_valid)[:, 1]

        auc = roc_auc_score(y_valid, proba)
        aucs.append(auc)

        trial.report(np.mean(aucs), step=fold)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(aucs)

study = optuna.create_study(direction="maximize", pruner=optuna.pruners.MedianPruner(n_warmup_steps=2))
study.optimize(lambda trial: objective(trial, X_train, y_train), n_trials=30, n_jobs=-1, show_progress_bar=True)


print("Best trial:")
print(study.best_trial.value)
print(study.best_trial.params)

[I 2026-05-11 12:44:31,404] A new study created in memory with name: no-name-db190817-6948-4a8e-87aa-c130fdcbba45
Best trial: 6. Best value: 0.883905:   0%|                                                     | 0/30 [07:10<?, ?it/s]

[I 2026-05-11 12:51:41,099] Trial 6 finished with value: 0.8839054211779841 and parameters: {'C': 0.0016358117361472825}. Best is trial 6 with value: 0.8839054211779841.


Best trial: 0. Best value: 0.886305:   7%|██▊                                       | 2/30 [07:46<1:32:35, 198.41s/it]

[I 2026-05-11 12:52:16,811] Trial 0 finished with value: 0.8863046675032328 and parameters: {'C': 0.0023789673048510043}. Best is trial 0 with value: 0.8863046675032328.


Best trial: 0. Best value: 0.886305:   7%|██▊                                       | 2/30 [08:06<1:32:35, 198.41s/it]

...

[I 2026-05-11 13:45:56,869] Trial 27 finished with value: 0.8935791390466763 and parameters: {'C': 3.3668321564182957}. Best is trial 1 with value: 0.8936401285242039.


Best trial: 1. Best value: 0.89364:  97%|█████████████████████████████████████████▌ | 29/30 [1:04:12<01:17, 77.99s/it]

[I 2026-05-11 13:48:44,387] Trial 28 finished with value: 0.8935380631601776 and parameters: {'C': 8.345209904579416}. Best is trial 1 with value: 0.8936401285242039.


Best trial: 1. Best value: 0.89364: 100%|██████████████████████████████████████████| 30/30 [1:05:33<00:00, 131.13s/it]

[I 2026-05-11 13:50:05,372] Trial 29 finished with value: 0.8935148391304348 and parameters: {'C': 7.775222795693056}. Best is trial 1 with value: 0.8936401285242039.
Best trial:
0.8936401285242039
{'C': 2.157929693953863}


In [15]:
model_tuned = make_pipeline(
    column_transformer,
    LogisticRegression(
        **study.best_params,
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    )
).fit(X_train, y_train.PitNextLap)

# Deploy

In [16]:
dump_pickle(model_tuned, '../models/model_logistic_regression.pkl')